# US Revenue Forecast 결과 조회

ticker, forecast_date, indicator를 입력받아 해당 데이터를 조회 및 출력하는 노트북입니다.

## 1. 라이브러리 임포트 및 설정

In [1]:
from pathlib import Path
import logging
import sys

# ---------------------------------------------------------
# 기본 로깅 설정
# ---------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)


# ---------------------------------------------------------
# 0) 프로젝트 루트 자동 탐색 (DATA 폴더 기준)
# ---------------------------------------------------------
def add_repo_path():
    """프로젝트 루트를 자동 탐색하여 sys.path에 추가"""
    if '__file__' in globals():
        current = Path(__file__).resolve().parent
    else:
        current = Path.cwd()

    for parent in [current] + list(current.parents):
        if (parent / "DATA").exists():
            if str(parent) not in sys.path:
                sys.path.insert(0, str(parent))
            logger.info(f"Project root added: {parent}")
            return str(parent)

    fallback = r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast"
    if os.path.isdir(fallback):
        if fallback not in sys.path:
            sys.path.insert(0, fallback)
        logger.warning(f"Using fallback path: {fallback}")
        return fallback

    raise FileNotFoundError("DATA 폴더를 찾을 수 없습니다.")


try:
    project_root = add_repo_path()
    from DATA.stock_invest_function import get_db_host
except ImportError:
    logger.warning("stock_invest_function import 실패 - DB 정보를 직접 설정해야 합니다")


2025-12-24 23:00:55 [INFO] Project root added: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
2025-12-24 23:01:03 [WARNING] From C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\keras\src\losses.py:2976: The name tf.losses.sparse_softmax_cross_entropy is deprecated. Please use tf.compat.v1.losses.sparse_softmax_cross_entropy instead.



In [2]:
import sys
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine, text
import warnings
from DATA.stock_invest_function import get_db_host
from typing import Optional
import os

warnings.filterwarnings('ignore')

print("✓ 라이브러리 임포트 완료")

✓ 라이브러리 임포트 완료


## 2. DB 연결 설정

아래 셀에서 DB 연결 정보를 수정하세요.

In [3]:
# DB 연결 정보 설정
db_config = {
    'host': get_db_host(),  # 실행 시 get_db_host()로 설정
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

print("✓ DB 설정 완료")
print(f"  Host: {db_config['host']}:{db_config['port']}")
print(f"  Database: {db_config['database']}")

✓ DB 설정 완료
  Host: 192.168.0.230:3307
  Database: investar


## 3. RevenueForecastViewer 클래스 정의

In [4]:
class RevenueForecastViewer:
    """Revenue Forecast 데이터 조회 클래스"""

    def __init__(self, db_config):
        """
        Args:
            db_config: DB 연결 정보 딕셔너리
                - host, port, database, user, password
        """
        self.db_config = db_config
        self.engine = None
        self._connect_db()

    def _connect_db(self):
        """DB 연결"""
        try:
            conn_str = (
                f"mysql+pymysql://{self.db_config['user']}:{self.db_config['password']}@"
                f"{self.db_config['host']}:{self.db_config['port']}/"
                f"{self.db_config['database']}?charset=utf8mb4"
            )
            self.engine = create_engine(conn_str)
            print(f"✓ DB 연결 성공: {self.db_config['host']}:{self.db_config['port']}/{self.db_config['database']}")
        except Exception as e:
            print(f"✗ DB 연결 실패: {e}")
            raise

    def get_available_tickers(self):
        """사용 가능한 ticker 목록 조회"""
        query = """
        SELECT DISTINCT ticker
        FROM us_revenue_forecast_result
        ORDER BY ticker
        """
        try:
            with self.engine.connect() as conn:
                df = pd.read_sql(query, conn)
            return df['ticker'].tolist()
        except Exception as e:
            print(f"✗ Ticker 목록 조회 실패: {e}")
            return []

    def get_available_forecast_dates(self, ticker=None):
        """사용 가능한 forecast_date 목록 조회"""
        if ticker:
            query = """
            SELECT DISTINCT forecast_date
            FROM us_revenue_forecast_result
            WHERE ticker = :ticker
            ORDER BY forecast_date DESC
            """
            params = {'ticker': ticker}
        else:
            query = """
            SELECT DISTINCT forecast_date
            FROM us_revenue_forecast_result
            ORDER BY forecast_date DESC
            """
            params = {}

        try:
            with self.engine.connect() as conn:
                if params:
                    df = pd.read_sql(text(query), conn, params=params)
                else:
                    df = pd.read_sql(query, conn)
            return df['forecast_date'].tolist()
        except Exception as e:
            print(f"✗ Forecast date 목록 조회 실패: {e}")
            return []

    def get_available_indicators(self, ticker=None, forecast_date=None):
        """사용 가능한 indicator 목록 조회"""
        conditions = []
        params = {}

        if ticker:
            conditions.append("ticker = :ticker")
            params['ticker'] = ticker
        if forecast_date:
            conditions.append("forecast_date = :forecast_date")
            params['forecast_date'] = forecast_date

        where_clause = " WHERE " + " AND ".join(conditions) if conditions else ""

        query = f"""
        SELECT DISTINCT indicator
        FROM us_revenue_forecast_result
        {where_clause}
        ORDER BY indicator
        """

        try:
            with self.engine.connect() as conn:
                if params:
                    df = pd.read_sql(text(query), conn, params=params)
                else:
                    df = pd.read_sql(query, conn)
            return df['indicator'].tolist()
        except Exception as e:
            print(f"✗ Indicator 목록 조회 실패: {e}")
            return []

    def query_revenue_forecast(self, ticker, forecast_date, indicator=None):
        """
        Revenue forecast 데이터 조회

        Args:
            ticker: 종목 코드
            forecast_date: 예측 날짜 (YYYY-MM-DD)
            indicator: 지표명 (None이면 모든 지표)

        Returns:
            DataFrame
        """
        conditions = ["ticker = :ticker", "forecast_date = :forecast_date"]
        params = {'ticker': ticker, 'forecast_date': forecast_date}

        if indicator:
            conditions.append("indicator = :indicator")
            params['indicator'] = indicator

        where_clause = " AND ".join(conditions)

        query = f"""
        SELECT
            date,
            ticker,
            indicator,
            value,
            forecast_date,
            created_at,
            updated_at
        FROM us_revenue_forecast_result
        WHERE {where_clause}
        ORDER BY date, indicator
        """

        try:
            with self.engine.connect() as conn:
                df = pd.read_sql(text(query), conn, params=params)

            if df.empty:
                print(f"⚠ 조회된 데이터가 없습니다.")
                return df

            # 날짜 형식 변환
            df['date'] = pd.to_datetime(df['date'])
            df['forecast_date'] = pd.to_datetime(df['forecast_date'])

            return df

        except Exception as e:
            print(f"✗ 데이터 조회 실패: {e}")
            return pd.DataFrame()

    def display_results(self, df, output_format='wide'):
        """
        조회 결과 출력

        Args:
            df: 조회된 DataFrame
            output_format: 'wide' (pivot 테이블) 또는 'long' (원본 형태)
        """
        if df.empty:
            return

        print("\n" + "=" * 100)
        print(f"조회 결과: {len(df)} rows")
        print("=" * 100)

        if output_format == 'wide':
            # Wide format (indicator별로 컬럼 분리)
            pivot_df = df.pivot_table(
                index='date',
                columns='indicator',
                values='value',
                aggfunc='first'
            )
            pivot_df.index = pivot_df.index.strftime('%Y-%m-%d')

            print("\n[Wide Format - Pivot Table]")
            display(pivot_df)

            # 통계 정보
            print("\n[통계 정보]")
            display(pivot_df.describe())

        else:
            # Long format (원본)
            print("\n[Long Format - Original Data]")
            display_df = df.copy()
            display_df['date'] = display_df['date'].dt.strftime('%Y-%m-%d')
            display_df['forecast_date'] = display_df['forecast_date'].dt.strftime('%Y-%m-%d')
            display(display_df)

        # 메타 정보
        print("\n" + "-" * 100)
        print(f"Ticker: {df['ticker'].iloc[0]}")
        print(f"Forecast Date: {df['forecast_date'].iloc[0].strftime('%Y-%m-%d')}")
        print(f"Date Range: {df['date'].min().strftime('%Y-%m-%d')} ~ {df['date'].max().strftime('%Y-%m-%d')}")
        print(f"Indicators: {', '.join(df['indicator'].unique())}")
        print("=" * 100)

    def save_to_csv(self, df, filename=None):
        """결과를 CSV 파일로 저장"""
        if df.empty:
            return

        if filename is None:
            ticker = df['ticker'].iloc[0]
            forecast_date = df['forecast_date'].iloc[0].strftime('%Y%m%d')
            filename = f"revenue_forecast_{ticker}_{forecast_date}.csv"

        try:
            df.to_csv(filename, index=False, encoding='utf-8-sig')
            print(f"\n✓ CSV 파일 저장 완료: {filename}")
        except Exception as e:
            print(f"\n✗ CSV 파일 저장 실패: {e}")

print("✓ RevenueForecastViewer 클래스 정의 완료")

✓ RevenueForecastViewer 클래스 정의 완료


## 4. Viewer 초기화

In [5]:
# Viewer 객체 생성
viewer = RevenueForecastViewer(db_config)

✓ DB 연결 성공: 192.168.0.230:3307/investar


## 5. 사용 가능한 데이터 목록 조회

### 5.1 Ticker 목록

In [6]:
# 사용 가능한 ticker 목록 조회
tickers = viewer.get_available_tickers()
print(f"\n사용 가능한 Ticker: {len(tickers)}개")
print("-" * 50)
print(tickers[:20])  # 처음 20개만 출력


사용 가능한 Ticker: 844개
--------------------------------------------------
['A', 'AAP', 'AAPL', 'ABBV', 'ABG', 'ABM', 'ABNB', 'ABT', 'ACA', 'ACAD', 'ACHC', 'ACIW', 'ACLS', 'ACMR', 'ACN', 'ACT', 'ADBE', 'ADEA', 'ADI', 'ADM']


### 5.2 Forecast Date 목록

In [7]:
# 사용 가능한 forecast_date 목록 조회
dates = viewer.get_available_forecast_dates()
print(f"\n사용 가능한 Forecast Date: {len(dates)}개")
print("-" * 50)
for date in dates:
    print(date)


사용 가능한 Forecast Date: 5개
--------------------------------------------------
2025-12-13
2025-11-19
2025-11-06
2025-11-04
2025-11-02


### 5.3 Indicator 목록

In [8]:
# 사용 가능한 indicator 목록 조회
indicators = viewer.get_available_indicators()
print(f"\n사용 가능한 Indicator: {len(indicators)}개")
print("-" * 50)
for i, indicator in enumerate(indicators, 1):
    print(f"{i:2d}. {indicator}")


사용 가능한 Indicator: 4개
--------------------------------------------------
 1. revenue_billions_esq_forecast
 2. revenue_billions_lstm_forecast
 3. revenue_billions_prophet_forecast
 4. revenue_billions_sarima_noexog


## 6. 데이터 조회

### 6.1 기본 조회 (모든 지표)

In [9]:
# 조회할 ticker와 forecast_date 설정
ticker = 'MU'              # 여기를 원하는 ticker로 수정
forecast_date = '2025-12-13'  # 여기를 원하는 날짜로 수정

# 데이터 조회
df = viewer.query_revenue_forecast(ticker, forecast_date)

# 결과 출력 (Wide format)
if not df.empty:
    viewer.display_results(df, output_format='wide')


조회 결과: 247 rows

[Wide Format - Pivot Table]


indicator,revenue_billions_esq_forecast,revenue_billions_lstm_forecast,revenue_billions_prophet_forecast,revenue_billions_sarima_noexog
date,,,,
2011-05-31,2.140000,NaN,2.140000,2.140000
2011-08-31,2.140000,2.140000,2.140000,2.140000
2011-11-30,2.090000,2.090000,2.090000,2.090000
2012-02-29,2.070000,2.070000,2.070000,2.070000
2012-05-31,2.170000,2.170000,2.170000,2.170000
...,...,...,...,...
2025-08-31,11.310000,11.310000,11.310000,11.310000
2025-11-30,12.812249,4.714456,9.113009,11.697452
2026-02-28,14.314498,4.832414,8.971951,11.618880



[통계 정보]


indicator,revenue_billions_esq_forecast,revenue_billions_lstm_forecast,revenue_billions_prophet_forecast,revenue_billions_sarima_noexog
count,62.000000,61.000000,62.000000,62.000000
mean,5.788427,5.165338,5.427044,5.584373
std,3.315946,2.173646,2.435708,2.753327
min,1.830000,1.830000,1.830000,1.830000
25%,3.705000,3.750000,3.705000,3.705000
50%,4.835000,4.730000,4.835000,4.835000
75%,7.622500,6.800000,7.622500,7.622500
max,17.318997,11.310000,11.310000,12.307327



----------------------------------------------------------------------------------------------------
Ticker: MU
Forecast Date: 2025-12-13
Date Range: 2011-05-31 ~ 2026-08-31
Indicators: revenue_billions_esq_forecast, revenue_billions_prophet_forecast, revenue_billions_sarima_noexog, revenue_billions_lstm_forecast


In [14]:
df[df['indicator'] == 'revenue_billions_esq_forecast'].tail(10)

,date,ticker,indicator,value,forecast_date,created_at,updated_at
207,2024-05-31,MU,revenue_billions_esq_forecast,6.810000,2025-12-13,2025-12-13 09:13:24,2025-12-13 18:13:35
211,2024-08-31,MU,revenue_billions_esq_forecast,7.750000,2025-12-13,2025-12-13 09:13:24,2025-12-13 18:13:35
215,2024-11-30,MU,revenue_billions_esq_forecast,8.710000,2025-12-13,2025-12-13 09:13:24,2025-12-13 18:13:35
219,2025-02-28,MU,revenue_billions_esq_forecast,8.050000,2025-12-13,2025-12-13 09:13:24,2025-12-13 18:13:35
223,2025-05-31,MU,revenue_billions_esq_forecast,9.300000,2025-12-13,2025-12-13 09:13:24,2025-12-13 18:13:35
227,2025-08-31,MU,revenue_billions_esq_forecast,11.310000,2025-12-13,2025-12-13 09:13:24,2025-12-13 18:13:35
231,2025-11-30,MU,revenue_billions_esq_forecast,12.812249,2025-12-13,2025-12-13 09:13:24,2025-12-13 18:13:35
235,2026-02-28,MU,revenue_billions_esq_forecast,14.314498,2025-12-13,2025-12-13 09:13:24,2025-12-13 18:13:35
239,2026-05-31,MU,revenue_billions_esq_forecast,15.816748,2025-12-13,2025-12-13 09:13:24,2025-12-13 18:13:35
243,2026-08-31,MU,revenue_billions_esq_forecast,17.318997,2025-12-13,2025-12-13 09:13:24,2025-12-13 18:13:35


### 6.2 특정 지표만 조회

In [15]:
# 조회할 ticker, forecast_date, indicator 설정
ticker = 'MU'
forecast_date = '2025-12-13'
indicator = 'revenue_billions_esq_forecast'  # 원하는 지표로 수정

 # 1. revenue_billions_esq_forecast
 # 2. revenue_billions_lstm_forecast
 # 3. revenue_billions_prophet_forecast
 # 4. revenue_billions_sarima_noexog

# 데이터 조회
df = viewer.query_revenue_forecast(ticker, forecast_date, indicator)

# 결과 출력 (Long format)
if not df.empty:
    viewer.display_results(df, output_format='long')


조회 결과: 63 rows

[Long Format - Original Data]


,date,ticker,indicator,value,forecast_date,created_at,updated_at
0,2011-03-31,FORM,revenue_billions_esq_forecast,0.040000,2025-12-13,2025-12-13 10:08:11,2025-12-13 19:08:21
1,2011-06-30,FORM,revenue_billions_esq_forecast,0.050000,2025-12-13,2025-12-13 10:08:11,2025-12-13 19:08:21
2,2011-09-30,FORM,revenue_billions_esq_forecast,0.050000,2025-12-13,2025-12-13 10:08:11,2025-12-13 19:08:21
3,2011-12-31,FORM,revenue_billions_esq_forecast,0.030000,2025-12-13,2025-12-13 10:08:11,2025-12-13 19:08:21
4,2012-03-31,FORM,revenue_billions_esq_forecast,0.030000,2025-12-13,2025-12-13 10:08:11,2025-12-13 19:08:21
...,...,...,...,...,...,...,...
58,2025-09-30,FORM,revenue_billions_esq_forecast,0.200000,2025-12-13,2025-12-13 10:08:11,2025-12-13 19:08:21
59,2025-12-31,FORM,revenue_billions_esq_forecast,0.202419,2025-12-13,2025-12-13 10:08:11,2025-12-13 19:08:21
60,2026-03-31,FORM,revenue_billions_esq_forecast,0.205141,2025-12-13,2025-12-13 10:08:11,2025-12-13 19:08:21
61,2026-06-30,FORM,revenue_billions_esq_forecast,0.207863,2025-12-13,2025-12-13 10:08:11,2025-12-13 19:08:21



----------------------------------------------------------------------------------------------------
Ticker: FORM
Forecast Date: 2025-12-13
Date Range: 2011-03-31 ~ 2026-09-30
Indicators: revenue_billions_esq_forecast


In [16]:
df.tail(10)

,date,ticker,indicator,value,forecast_date,created_at,updated_at
53,2024-06-30,FORM,revenue_billions_esq_forecast,0.200000,2025-12-13,2025-12-13 10:08:11,2025-12-13 19:08:21
54,2024-09-30,FORM,revenue_billions_esq_forecast,0.210000,2025-12-13,2025-12-13 10:08:11,2025-12-13 19:08:21
55,2024-12-31,FORM,revenue_billions_esq_forecast,0.190000,2025-12-13,2025-12-13 10:08:11,2025-12-13 19:08:21
56,2025-03-31,FORM,revenue_billions_esq_forecast,0.170000,2025-12-13,2025-12-13 10:08:11,2025-12-13 19:08:21
57,2025-06-30,FORM,revenue_billions_esq_forecast,0.200000,2025-12-13,2025-12-13 10:08:11,2025-12-13 19:08:21
58,2025-09-30,FORM,revenue_billions_esq_forecast,0.200000,2025-12-13,2025-12-13 10:08:11,2025-12-13 19:08:21
59,2025-12-31,FORM,revenue_billions_esq_forecast,0.202419,2025-12-13,2025-12-13 10:08:11,2025-12-13 19:08:21
60,2026-03-31,FORM,revenue_billions_esq_forecast,0.205141,2025-12-13,2025-12-13 10:08:11,2025-12-13 19:08:21
61,2026-06-30,FORM,revenue_billions_esq_forecast,0.207863,2025-12-13,2025-12-13 10:08:11,2025-12-13 19:08:21
62,2026-09-30,FORM,revenue_billions_esq_forecast,0.210585,2025-12-13,2025-12-13 10:08:11,2025-12-13 19:08:21


In [17]:
def get_unique_values_from_db(db_info):
    """
    investar.us_psr_valuation_result 테이블에서
    forecast_date.unique(), indicator.unique() 값을 리스트로 반환
    """
    # ✅ DB 연결
    conn = pymysql.connect(
        host=db_info['host'],
        port=db_info['port'],
        user=db_info['user'],
        password=db_info['password'],
        database=db_info['database'],
        charset='utf8mb4'
    )

    try:
        # ✅ forecast_date unique 값 추출
        query_forecast = """
        SELECT DISTINCT forecast_date
        FROM us_psr_valuation_result
        ORDER BY forecast_date;
        """
        df_forecast = pd.read_sql(query_forecast, conn)
        forecast_dates = df_forecast['forecast_date'].astype(str).tolist()

        # ✅ indicator unique 값 추출
        query_indicator = """
        SELECT DISTINCT indicator
        FROM us_psr_valuation_result
        ORDER BY indicator;
        """
        df_indicator = pd.read_sql(query_indicator, conn)
        indicators = df_indicator['indicator'].tolist()

        return forecast_dates, indicators

    finally:
        conn.close()

def get_indicator_pivot(db_info: dict,
                        indicator: str = "sarima_valuation",
                        forecast_date: Optional[str] = None) -> pd.DataFrame:
    """
    us_psr_valuation_result에서 특정 indicator의 값을 불러와
    index=date, columns=ticker, values=value 피벗 테이블을 반환.

    Args:
        db_info: {'host','port','user','password','database'}
        indicator: 예) 'sarima_valuation'
        forecast_date: 'YYYY-MM-DD' 문자열. None이면 최신 forecast_date 자동 선택.

    Returns:
        pd.DataFrame: 날짜 × 티커 피벗 (값=indicator의 value)
    """
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4",
        autocommit=True,
    )

    try:
        # 1️⃣ 최신 forecast_date 자동 선택
        if forecast_date is None:
            q_latest = """
                SELECT MAX(forecast_date) AS latest_fd
                FROM us_psr_valuation_result
                WHERE indicator = %s
            """
            latest_fd = pd.read_sql(q_latest, conn, params=[indicator])["latest_fd"][0]
            if pd.isna(latest_fd):
                raise ValueError(f"No rows found for indicator='{indicator}'.")
            forecast_date = str(latest_fd)

        # 2️⃣ 해당 forecast_date 자료 조회
        q = """
            SELECT `date`, `ticker`, `value`, `updated_ts`
            FROM us_psr_valuation_result
            WHERE indicator = %s AND forecast_date = %s
            ORDER BY `updated_ts` ASC
        """
        df = pd.read_sql(q, conn, params=[indicator, forecast_date])

        if df.empty:
            raise ValueError(f"No rows for indicator='{indicator}' on forecast_date='{forecast_date}'.")

        # 3️⃣ 중복 제거 (최신 updated_ts만)
        df = df.sort_values("updated_ts").drop_duplicates(subset=["date", "ticker"], keep="last")

        # 4️⃣ Pivot
        df["date"] = pd.to_datetime(df["date"])
        pivot = df.pivot(index="date", columns="ticker", values="value").sort_index()
        pivot = pivot.sort_index(axis=1)

        pivot.index.name = "date"
        pivot.columns.name = None

        print(f"✅ Loaded indicator='{indicator}', forecast_date={forecast_date}, shape={pivot.shape}")
        return pivot

    finally:
        conn.close()


def rank_changes_between(df_pivot: pd.DataFrame,
                         start_date: str = "2025-10-30",
                         end_date: str = "2026-12-31",
                         fill: str = "ffill") -> pd.DataFrame:
    """
    기간 [start_date, end_date] 내에서 각 컬럼의 변화율을 계산해 내림차순 정렬.

    변화율 정의: (마지막값 / 처음값 - 1) * 100 (%)

    Args:
        df_pivot : index=DatetimeIndex, columns=tickers
        start_date, end_date : 'YYYY-MM-DD' 문자열
        fill : 결측 처리 방식
               - "none": 원본 그대로(결측 있으면 해당 컬럼 NaN 결과)
               - "ffill": 기간 내 결측을 앞/뒤로 보간(ffill 후 bfill)

    Returns:
        pd.DataFrame(columns=["start_value","end_value","abs_change","pct_change_%"])
        인덱스=티커, pct_change_% 기준 내림차순 정렬
    """
    df = df_pivot.copy()
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    # 1) 구간 슬라이스 (시작/끝 날짜가 정확히 없더라도 범위에 맞는 행 선택됨)
    period = df.loc[start_date:end_date]
    if period.empty:
        raise ValueError(f"No rows between {start_date} and {end_date}.")

    # 2) 결측 처리
    if fill == "ffill":
        period = period.ffill().bfill()
    elif fill == "none":
        pass
    else:
        raise ValueError("fill must be one of {'none','ffill'}")

    # 3) 각 컬럼의 처음/마지막 유효값 추출
    def first_valid(s: pd.Series):
        v = s.dropna()
        return v.iloc[0] if not v.empty else np.nan

    def last_valid(s: pd.Series):
        v = s.dropna()
        return v.iloc[-1] if not v.empty else np.nan

    first_vals = period.apply(first_valid, axis=0)
    last_vals  = period.apply(last_valid, axis=0)

    # 4) 변화율/절대변화 계산
    out = pd.DataFrame({
        "start_value": first_vals,
        "end_value": last_vals,
        "abs_change": last_vals - first_vals,
        "pct_change_%": (last_vals / first_vals - 1.0) * 100.0
    })

    # 시작/끝값이 없는 컬럼 제거
    out = out.dropna(subset=["start_value", "end_value"])

    # 5) 변화율 내림차순 정렬
    out = out.sort_values("pct_change_%", ascending=False)

    return out


# ── 사용 예시 ───────────────────────────────
# db_info = {
#     'host': 'localhost',
#     'port': 3307,
#     'user': 'stox7412',
#     'password': 'Apt106503!~',
#     'database': 'investar'
# }
# df_pivot = get_indicator_pivot(db_info, indicator='sarima_valuation')
# display(df_pivot.head())


# ── 사용 예시 ─────────────────────────────────────────
# 결과 표 (상승률 높은 순)
# rank_df = rank_changes_between(df_pivot,
#                                start_date="2025-10-30",
#                                end_date="2026-12-31",
#                                fill="ffill")   # 결측이 많다면 'ffill' 권장
# print(rank_df.head(20))        # 상위 20개 확인

In [18]:
import pymysql

forecast_dates, indicators = get_unique_values_from_db(db_config)
print("📅 forecast_date.unique():", forecast_dates)
print("📊 indicator.unique():", indicators)

📅 forecast_date.unique(): ['2025-11-06', '2025-11-19']
📊 indicator.unique(): ['es_valuation', 'lstm_valuation', 'prophet_valuation', 'PSR_es_forecast', 'PSR_prophet_forecast_noexog', 'PSR_ttm_lstm_forecast', 'PSR_ttm_sarima_forecast', 'revenue_billions_avg_of_4_ttm', 'revenue_billions_esq_forecast_ttm', 'revenue_billions_lstm_forecast_ttm', 'revenue_billions_prophet_forecast_ttm', 'revenue_billions_sarima_noexog_ttm', 'sarima_valuation']


In [19]:
df_pivot = get_indicator_pivot(db_config, indicator='es_valuation')

✅ Loaded indicator='es_valuation', forecast_date=2025-11-19, shape=(39, 843)


In [20]:
psr_pivot = get_indicator_pivot(db_config, indicator='PSR_ttm_sarima_forecast')
psr_pivot[ticker].tail(36)

✅ Loaded indicator='PSR_ttm_sarima_forecast', forecast_date=2025-11-19, shape=(39, 843)


date
2024-05-31         NaN
2024-06-30         NaN
2024-07-31         NaN
2024-08-31         NaN
2024-09-30         NaN
2024-10-31         NaN
2024-11-30         NaN
2024-12-31         NaN
2025-01-31         NaN
2025-02-28         NaN
2025-03-31         NaN
2025-04-30         NaN
2025-05-31    3.121622
2025-06-30    3.694444
2025-07-31    3.128571
2025-08-31    3.169014
2025-09-30    3.797297
2025-10-31    5.519481
2025-11-30    5.463553
2025-12-31    5.408521
2026-01-31    5.354367
2026-02-28    5.301072
2026-03-31    5.248622
2026-04-30    5.196998
2026-05-31    5.146185
2026-06-30    5.096167
2026-07-31    5.046930
2026-08-31    4.998457
2026-09-30    4.950735
2026-10-31    4.903749
2026-11-30    4.857485
2026-12-31    4.811930
2027-01-31    4.767069
2027-02-28    4.722891
2027-03-31    4.679381
2027-04-30    4.636529
Name: FORM, dtype: float64

In [26]:
ttm_pivot = get_indicator_pivot(db_config, indicator='revenue_billions_sarima_noexog_ttm')
ttm_pivot[ticker].tail(36)

✅ Loaded indicator='revenue_billions_sarima_noexog_ttm', forecast_date=2025-11-19, shape=(39, 843)


date
2024-05-31         NaN
2024-06-30         NaN
2024-07-31         NaN
2024-08-31         NaN
2024-09-30         NaN
2024-10-31         NaN
2024-11-30         NaN
2024-12-31         NaN
2025-01-31         NaN
2025-02-28         NaN
2025-03-31         NaN
2025-04-30         NaN
2025-05-31    0.770000
2025-06-30    0.770000
2025-07-31    0.770000
2025-08-31    0.770000
2025-09-30    0.760000
2025-10-31    0.760000
2025-11-30    0.760000
2025-12-31    0.772360
2026-01-31    0.772360
2026-02-28    0.772360
2026-03-31    0.807107
2026-04-30    0.807107
2026-05-31    0.807107
2026-06-30    0.814269
2026-07-31    0.814269
2026-08-31    0.814269
2026-09-30    0.823876
2026-10-31    0.823876
2026-11-30    0.823876
2026-12-31    0.833597
2027-01-31    0.833597
2027-02-28    0.833597
2027-03-31    0.843431
2027-04-30    0.843431
Name: FORM, dtype: float64

In [27]:
method = 'revenue_billions_sarima_noexog_ttm'

# date = pd

rank_df = rank_changes_between(ttm_pivot,
                               start_date="2025-11-30",
                               end_date="2026-12-31",
                               fill="ffill")   # 결측이 많다면 'ffill' 권장

today_date = datetime.today().strftime("%Y-%m-%d")
file_name = f"{method}_{today_date}.xlsx"

save_dir = r"C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, file_name)

# =========================
# Excel 저장
# =========================
rank_df.to_excel(save_path, index=True)

print(f"저장 완료: {save_path}")

저장 완료: C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\revenue_billions_sarima_noexog_ttm_2025-12-18.xlsx


In [25]:
rank_df

,start_value,end_value,abs_change,pct_change_%
NX,-0.112506,-5.789601,-5.677095,5046.026354
FMC,-0.496817,-23.266036,-22.769219,4583.021308
BG,17.370206,77.370389,60.000183,345.420099
LUMN,14.117218,60.611289,46.494071,329.343002
SFNC,0.868346,3.649828,2.781482,320.319453
...,...,...,...,...
THRY,0.124815,-2.334330,-2.459145,-1970.234833
OGN,0.685218,-13.087766,-13.772984,-2010.013524
AUB,1.193509,-23.439790,-24.633299,-2063.939490
VSTS,0.089093,-2.197796,-2.286889,-2566.854130


In [58]:
df_pivot['FORM']

date
2024-02-29          NaN
2024-03-31          NaN
2024-04-30          NaN
2024-05-31          NaN
2024-06-30          NaN
2024-07-31          NaN
2024-08-31          NaN
2024-09-30          NaN
2024-10-31          NaN
2024-11-30          NaN
2024-12-31          NaN
2025-01-31          NaN
2025-02-28          NaN
2025-03-31          NaN
2025-04-30          NaN
2025-05-31     2.403649
2025-06-30     2.844722
2025-07-31     2.409000
2025-08-31     2.440141
2025-09-30     2.885946
2025-10-31     4.194805
2025-11-30     5.190358
2025-12-31     6.293047
2026-01-31     7.305842
2026-02-28     8.318637
2026-03-31     9.769171
2026-04-30    10.829477
2026-05-31    11.889782
2026-06-30    13.100089
2026-07-31    14.172676
2026-08-31    15.245263
2026-09-30    16.566603
2026-10-31    17.655541
2026-11-30    18.744479
2026-12-31    20.046164
2027-01-31    21.146783
2027-02-28    22.247401
2027-03-31    23.579017
2027-04-30    24.690525
Name: FORM, dtype: float64

In [34]:
value_pivot = get_indicator_pivot(db_config, indicator='prophet_valuation')
value_pivot.tail(12)

method = 'prophet_valuation'

# date = pd

rank_df = rank_changes_between(value_pivot,
                               start_date="2025-11-30",
                               end_date="2026-12-31",
                               fill="ffill")   # 결측이 많다면 'ffill' 권장

today_date = datetime.today().strftime("%Y-%m-%d")
file_name = f"{method}_{today_date}.xlsx"

save_dir = r"C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, file_name)

# =========================
# Excel 저장
# =========================
rank_df.to_excel(save_path, index=True)

print(f"저장 완료: {save_path}")

✅ Loaded indicator='prophet_valuation', forecast_date=2025-11-19, shape=(39, 843)
저장 완료: C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\prophet_valuation_2025-12-18.xlsx
